# Financial Asset Recommendation System

## Part 3 — Cleaning, formatting and temporal splitting

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd
from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.preprocessing import clean_data, temporal_user_split
assets=pd.read_csv(RAW_DATA_DIR/'assets.csv')
users=pd.read_csv(RAW_DATA_DIR/'users.csv')
interactions=pd.read_csv(RAW_DATA_DIR/'interactions.csv')

## 1. Validation and cleaning

The reusable `clean_data` function checks the required schema, referential integrity, duplicate identifiers, valid timestamps and positive implicit-feedback weights. Keeping these checks in reusable source code makes the preprocessing reproducible rather than notebook-specific.

### Synthetic data acquisition and statistical design

The project uses **synthetic but finance-shaped data** rather than claiming that the observations are real market behavior. Investor profiles, asset characteristics and implicit interactions are generated from controlled rules and probability distributions with fixed random seeds. Interaction propensity is influenced by variables such as sector preference, risk compatibility and asset characteristics, so the generated feedback contains a learnable signal while remaining fully reproducible.

This design is appropriate for demonstrating the recommendation pipeline, but it does **not** establish real-world investment effectiveness. The synthetic assumptions can make some relationships cleaner than they would be in production data; therefore, conclusions below are limited to recommendation quality on this generated dataset.


In [3]:
assets,users,interactions=clean_data(assets,users,interactions)
print(assets.shape,users.shape,interactions.shape)

(60, 9) (600, 6) (7731, 5)


## 2. Temporal user-level split

For every user, the most recent interaction is assigned to test, the preceding interaction to validation, and all earlier interactions to training. This prevents future behavior from leaking into model training.

In [4]:
train,validation,test=temporal_user_split(interactions,validation_items=1,test_items=1)
summary=pd.DataFrame({'rows':[len(train),len(validation),len(test)],'users':[train.user_id.nunique(),validation.user_id.nunique(),test.user_id.nunique()]},index=['train','validation','test'])
summary

,rows,users
train,6531,600
validation,600,600
test,600,600


In [5]:
check = pd.DataFrame({
    'train_max': train.groupby('user_id').timestamp.max(),
    'validation_time': validation.groupby('user_id').timestamp.min(),
    'test_time': test.groupby('user_id').timestamp.min()
})

chronology_ok = (
    (check.train_max <= check.validation_time) &
    (check.validation_time <= check.test_time)
)

print('Users respecting chronology:', chronology_ok.mean())

assert chronology_ok.all(), \
    'Temporal leakage detected for at least one user.'

# Check that the split preserves all interactions
assert len(train) + len(validation) + len(test) == len(interactions), \
    'Some interactions were lost or duplicated during splitting.'

# Each retained user should have validation and test interactions
assert set(train.user_id) == set(validation.user_id) == set(test.user_id), \
    'User sets are inconsistent between train, validation and test.'

print('Temporal split and leakage checks passed.')


Users respecting chronology: 1.0
Temporal split and leakage checks passed.


## 3. Save processed data

In [6]:
PROCESSED_DATA_DIR.mkdir(parents=True,exist_ok=True)
assets.to_csv(PROCESSED_DATA_DIR/'assets.csv',index=False)
users.to_csv(PROCESSED_DATA_DIR/'users.csv',index=False)
train.to_csv(PROCESSED_DATA_DIR/'train.csv',index=False)
validation.to_csv(PROCESSED_DATA_DIR/'validation.csv',index=False)
test.to_csv(PROCESSED_DATA_DIR/'test.csv',index=False)
print('Saved to',PROCESSED_DATA_DIR)

Saved to /home/marto/Desktop/SU_DeepLearning/DeepLearning/data/processed


## 4. Statistical validity and limitations

The chronological user-level split mirrors the intended recommendation setting: the model learns from earlier behavior and is evaluated on later behavior. This avoids the most direct form of future-interaction leakage and is more defensible than mixing future and past observations within a user's history.
